In [23]:
# 6-30-2026

In [24]:
# cretae transfer matrix with raw distance method:
#   for each domain descritpor vector, normalize and compute distance (cos/euc) between
#       it and all other domain vectors
#   turn distance into negative (greater=closer=better transferability)
# see corr between this matrix and ground truth matrix

In [25]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, kendalltau

In [26]:
descriptors = pd.read_csv("../encoder-inputs/domain_descriptions.csv")

In [27]:
descriptors.set_index("domain_id", inplace=True)
descriptors.index.name = None
descriptors.head()

,vpd_mean,vpd_std,vpd_p90,ws10_mean,ws10_std,ws10_p90,t2m_max_mean,t2m_max_std,t2m_max_p90,ndvi_mean,...,swvl4_p90,gwis_ba_mean,gwis_ba_std,gwis_ba_p90,cams_frpfire_mean,cams_frpfire_std,cams_frpfire_p90,fire_sparsity,land_cover_diversity,fire_season_length
0,11.229313,5.895861,19.746270,2.516127,1.052111,3.968697,302.43317,4.790887,307.46002,0.641714,...,0.479085,547.38410,1932.2546,1112.03370,0.067789,0.469590,0.097816,0.081824,0.728328,6
1,6.742952,3.144529,10.522034,1.406203,0.511182,2.132592,301.80650,4.735351,305.51370,0.796118,...,0.499585,338.66165,1157.0903,705.89197,0.063071,0.392797,0.108795,0.013560,0.218936,7
2,21.599873,10.565010,36.654785,4.066474,0.951055,5.291819,303.89624,6.380806,311.61792,0.242577,...,0.257699,3360.49340,8181.0723,8625.16300,0.162555,0.925012,0.219152,0.032987,0.602900,7
3,4.211150,2.716757,7.958644,3.166607,1.505368,5.099028,286.08792,10.954839,299.41890,0.550528,...,0.411259,328.76685,739.6544,901.44635,0.041860,0.274789,0.000000,0.001419,0.365885,3
4,2.564834,3.033652,6.938553,3.300411,1.380530,5.114551,273.42078,16.251247,293.80110,0.315671,...,0.447848,669.05206,2307.7700,1315.65490,0.185203,1.571907,0.090595,0.007227,0.615066,5


In [28]:
used_domains = [0, 1, 2, 4, 5, 6, 7, 8, 11, 12, 13, 16, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 32, 33, 36, 37, 38, 39, 45, 46, 47, 49]

In [29]:
len(used_domains)

34

In [30]:
test_domains = [45, 46, 47, 49]
train_domains = [d for d in used_domains if d not in test_domains]

active_descriptors = descriptors.loc[used_domains] # get only domains used in T

In [31]:
train_mean = active_descriptors.loc[train_domains].mean() # get only train domains, get mean of each col
train_std = active_descriptors.loc[train_domains].std() # same for std
# z-score norm
normalized = (active_descriptors - train_mean) / train_std

In [32]:
dist_matrix = squareform(pdist(normalized.values, metric="euclidean"))
dist_matrix = pd.DataFrame(dist_matrix, index=normalized.index, columns=normalized.index)

In [33]:
rawdist_matrix = -dist_matrix

In [34]:
rawdist_matrix.head(10)

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,-0.000000,-6.672423,-11.045133,-7.986460,-3.308284,-7.958821,-6.068376,-4.666705,-5.184888,-4.589527,...,-7.662378,-6.407206,-8.197987,-5.296501,-4.204545,-11.505974,-7.273316,-13.276326,-6.751253,-3.125536
1,-6.672423,-0.000000,-13.599319,-9.841238,-5.547623,-9.436429,-9.730264,-5.698760,-8.974480,-8.392720,...,-11.038897,-10.159206,-10.744922,-5.711799,-7.998925,-13.061897,-11.181847,-15.567050,-8.537365,-6.931799
2,-11.045133,-13.599319,-0.000000,-13.082436,-11.499412,-12.708249,-9.285989,-13.684210,-9.565427,-8.687225,...,-7.894825,-9.563264,-10.696453,-11.590126,-9.063832,-9.620238,-7.577379,-16.658596,-9.353973,-11.288658
4,-7.986460,-9.841238,-13.082436,-0.000000,-8.019986,-3.351193,-7.878595,-9.379059,-9.467587,-9.484902,...,-10.290864,-7.254905,-6.459036,-9.505884,-7.903673,-13.860378,-11.642435,-9.120748,-5.452087,-7.889157
5,-3.308284,-5.547623,-11.499412,-8.019986,-0.000000,-7.551036,-6.591061,-4.260023,-4.860621,-5.079144,...,-8.673302,-7.712744,-8.004912,-4.035172,-5.374218,-12.376469,-7.461498,-12.675088,-6.924344,-4.980700
6,-7.958821,-9.436429,-12.708249,-3.351193,-7.551036,-0.000000,-7.307292,-8.895403,-9.258151,-9.329919,...,-9.975772,-7.488843,-4.461359,-9.152761,-7.552179,-13.639737,-11.240952,-8.347249,-5.771723,-7.863910
7,-6.068376,-9.730264,-9.285989,-7.878595,-6.591061,-7.307292,-0.000000,-8.832793,-7.072502,-5.136909,...,-4.850997,-4.276811,-5.501850,-8.444666,-2.986468,-8.724586,-7.209533,-12.837734,-5.064828,-5.701705
8,-4.666705,-5.698760,-13.684210,-9.379059,-4.260023,-8.895403,-8.832793,-0.000000,-7.504786,-7.592999,...,-10.594288,-8.880334,-9.930977,-5.531485,-7.389167,-14.100157,-10.431665,-13.559093,-9.241698,-5.032279
11,-5.184888,-8.974480,-9.565427,-9.467587,-4.860621,-9.258151,-7.072502,-7.504786,-0.000000,-5.607953,...,-8.563762,-8.084519,-8.778158,-4.434500,-6.187654,-12.536701,-4.309561,-13.146039,-7.716434,-6.559332
12,-4.589527,-8.392720,-8.687225,-9.484902,-5.079144,-9.329919,-5.136909,-7.592999,-5.607953,-0.000000,...,-6.457259,-6.402053,-8.537121,-6.857717,-4.288140,-9.152598,-5.360677,-14.267618,-6.997054,-5.922538


In [35]:
true_matrix = pd.read_csv("../transfer-matrix/rf_transfer_matrix.csv")

In [36]:
true_matrix.set_index("Unnamed: 0", inplace=True)
true_matrix.index.name = None
true_matrix.columns = true_matrix.columns.astype(int)
true_matrix.head()

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.392727,0.291660,0.127238,0.058840,0.214817,0.084705,0.149056,0.182358,0.173877,0.026375,...,0.228505,0.243433,0.149609,0.275203,0.112525,-0.011557,0.116862,0.158796,0.025454,0.029928
1,0.142172,0.395760,0.030643,0.010376,0.180707,0.064432,0.056092,0.128808,0.097788,0.039590,...,0.167477,0.032806,0.029704,0.193033,0.055264,0.066550,0.108203,0.199762,0.023240,-0.037710
2,0.196600,0.214445,0.368173,0.023968,0.207323,0.140139,0.172990,0.231019,0.156741,0.057226,...,0.197402,0.223793,0.121457,0.248807,0.134485,0.053503,-0.005610,0.253965,0.031899,0.059566
4,0.235489,0.227173,0.127394,0.410382,0.215954,0.186207,0.125068,0.196603,0.165264,0.034514,...,0.178055,0.245668,0.147579,0.284360,0.217076,-0.003941,0.037643,0.306750,0.137154,0.020968
5,0.190803,0.206530,0.097887,0.087230,0.464604,0.144950,0.159361,0.204884,0.264641,0.104493,...,0.197195,0.232923,0.117228,0.292623,0.170687,-0.020694,0.115293,0.297271,0.067951,0.051770


In [37]:
true_matrix.shape == rawdist_matrix.shape # :)

True

In [ ]:
overall_corr, _ = spearmanr(rawdist_matrix.values.flatten(), true_matrix.values.flatten())
overall_corr
# low correlation, 0.117 for kendall and 0.171 for spearman

np.float64(0.17156122350971637)

In [ ]:
diag_rawdist = np.diag(rawdist_matrix.values)
diag_true = np.diag(true_matrix.values)
diag_corr, _ = kendalltau(diag_rawdist, diag_true)
diag_corr
# can't get diag corr because cannot compute for a all 0's diag

np.float64(nan)

In [ ]:
mask = ~np.eye(len(used_domains), dtype=bool)
offdiag_rawdist = rawdist_matrix.values[mask]
offdiag_true = true_matrix.values[mask]
offdiag_corr, _ = spearmanr(offdiag_rawdist, offdiag_true)
offdiag_corr
# almost no correlation

# rawdist diag is always 0 (self distance), true diag is always high (~0.4, indomain perf)
# this creates a fake consistent pattern across all 34 domains that spearman picks up
# offdiag removes these pairs, leaving the real cross domain signal, which is weaker
# so offdiag is the trustworthy number here, overall is inflated

np.float64(0.09428581943326335)

In [47]:
# 4x4 test submatrix
test_rawdist = rawdist_matrix.loc[test_domains, test_domains]
test_true = true_matrix.loc[test_domains, test_domains]
test_corr, _ = spearmanr(test_rawdist.values.flatten(), test_true.values.flatten())
test_corr

np.float64(0.607185884878083)

In [ ]:
mask_4x4 = ~np.eye(len(test_domains), dtype=bool)
offdiag_test_rawdist = test_rawdist.values[mask_4x4]
offdiag_test_true = test_true.values[mask_4x4]

offdiag_test_corr, _ = spearmanr(offdiag_test_rawdist, offdiag_test_true)
offdiag_test_corr
# trust offdiag numbers (~0.1 both), not overall (0.17 full, 0.6 on 4x4)
# 4x4 overall is most distorted: diag pairs are 4/16 entries here vs 34/~1100 in full
#   matrix, so the rawdist=0/true=high artifact dominates a small matrix way more
# offdiag-full and offdiag-4x4 agreeing at ~0.1 is the real signal: rawdist barely
#   predicts cross domain transfer, on its own or restricted to held out domains

np.float64(0.08481041912882634)

In [ ]:
assert (rawdist_matrix.index == true_matrix.index).all()
assert (rawdist_matrix.columns == true_matrix.columns).all()
# ok its good and no error with alignment

In [52]:
# low rawdist correlation means descriptor geometry alone isnt enouh

In [53]:
rawdist_matrix.to_csv("rawdist_matrix.csv")